# Tin tức (Unstructured Data) — One-layer Ingestion

Notebook điều khiển luồng ingestion 1 tầng cho 2 nguồn `rss`, `html`.

Mỗi nguồn ghi output riêng theo layout:
- `news/<source>/date=<run_date>/PART-000.parquet`

Partition theo ngày (date=YYYY-MM-DD). Khi rerun cùng ngày có thể truncate partition rồi ghi file mới.

Schema của output thống nhất theo `NEWS_COLUMNS`, phục vụ downstream sentiment analysis.

In [1]:
import os, sys, warnings, threading

os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

_orig_hook = threading.excepthook
def _quiet_hook(args):
    if "UnicodeDecodeError" in str(args.exc_type):
        pass
    else:
        _orig_hook(args)
threading.excepthook = _quiet_hook
warnings.filterwarnings("ignore")

from pathlib import Path
root = Path.cwd()
if not (root / "ingestion").is_dir():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from ingestion.common import (
    configure_logging,
    load_dotenv_from_project_root,
 )
from ingestion.unstructured_data import (
    NEWS_COLUMNS,
    NewsIngestionConfig,
    ingest_news,
    validate_news_df,
 )

configure_logging()
load_dotenv_from_project_root()
print("[OK] setup (RSS+HTML only mode)")

2026-05-13 14:34:33 [INFO] Đã nạp biến môi trường từ D:\WorkSpace\Đồ Án 2\stock-pipeline\.env


[OK] setup (RSS+HTML only mode)


## Cấu hình

In [2]:
rate_limit = int(os.getenv("NEWS_RATE_LIMIT_RPM", "60"))

news_cfg = NewsIngestionConfig(
    use_listing_tickers=True,
    listing_exchange_filter=["HSX", "HNX", "UPCOM"],
    max_tickers_per_run=50,
    max_articles_per_source=200,
    rss_max_per_feed=200,
    html_max_per_source=200,
    days_back=7,
    days_back_rss=7,
    days_back_html=7,
    strict_published_at_days_back=False,
    rate_limit_rpm=rate_limit,
    enable_rss=True,
    enable_html=True,
    enable_ticker_match=True,
    append_only=False,
    truncate_partition=True,
)

print(f"run_date     : {news_cfg.run_date}")
print(f"news_root    : {news_cfg.news_root}")
print(f"sources.yaml : {news_cfg.resolved_sources_yaml()}")
print(f"listing      : {news_cfg.resolved_listing_parquet()}")
print(f"listing_ok   : {news_cfg.resolved_listing_parquet().is_file()}")
print(f"tickers      : {len(news_cfg.resolved_tickers())}")
print(f"days_back    : rss={news_cfg.days_back_rss}d html={news_cfg.days_back_html}d (fallback={news_cfg.days_back}, strict={news_cfg.strict_published_at_days_back})")
print(f"max_per      : rss={news_cfg.rss_max_per_feed} html={news_cfg.html_max_per_source}")
print(f"focus_mode   : RSS+HTML only (rss_enabled={news_cfg.enable_rss} html_enabled={news_cfg.enable_html})")
print(f"rate_limit   : {news_cfg.rate_limit_rpm} rpm")
print(f"write_mode   : append_only={news_cfg.append_only} (truncate_partition={news_cfg.truncate_partition})")
print(f"schema       : {NEWS_COLUMNS}")

run_date     : 2026-05-13
news_root    : D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news
sources.yaml : D:\WorkSpace\Đồ Án 2\stock-pipeline\ingestion\unstructured_data\sources.yaml
listing      : D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\listing\master\listing.parquet
listing_ok   : True
tickers      : 30
days_back    : rss=7d html=7d (fallback=7, strict=False)
max_per      : rss=200 html=200
focus_mode   : RSS+HTML only (rss_enabled=True html_enabled=True)
rate_limit   : 60 rpm
write_mode   : append_only=False (truncate_partition=True)
schema       : ['article_id', 'source', 'ticker', 'title', 'summary', 'body_text', 'url', 'published_at', 'fetched_at', 'language', 'raw_ref']


## Chạy

In [3]:
news_paths = ingest_news(news_cfg)
news_paths

2026-05-13 14:34:33 [INFO] ingest_news mode: RSS+HTML only
2026-05-13 14:34:34 [INFO] RSS feed=https://vnexpress.net/rss/kinh-doanh.rss entries_found=60 rows_added=60 take<=200
2026-05-13 14:34:34 [INFO] RSS feed=https://vnexpress.net/rss/kinh-doanh/chung-khoan.rss entries_found=0 rows_added=0 take<=200
2026-05-13 14:34:38 [INFO] RSS feed=https://cafef.vn/home.rss entries_found=60 rows_added=60 take<=200
2026-05-13 14:34:40 [INFO] RSS feed=https://cafef.vn/xa-hoi.rss entries_found=50 rows_added=50 take<=200
2026-05-13 14:34:40 [INFO] RSS feed=https://cafef.vn/bat-dong-san.rss entries_found=50 rows_added=50 take<=200
2026-05-13 14:34:41 [INFO] RSS feed=https://cafef.vn/doanh-nghiep.rss entries_found=50 rows_added=50 take<=200
2026-05-13 14:34:42 [INFO] RSS feed=https://cafef.vn/tai-chinh-ngan-hang.rss entries_found=50 rows_added=50 take<=200
2026-05-13 14:34:43 [INFO] RSS feed=https://cafef.vn/tai-chinh-quoc-te.rss entries_found=50 rows_added=50 take<=200
2026-05-13 14:34:45 [INFO] RSS 

KeyboardInterrupt: 

## Kiểm tra kết quả

In [ ]:
import pandas as pd

print("=" * 70)
print("OUTPUT FILES")
print("=" * 70)
for key in ["rss", "html"]:
    info = news_paths.get(key, {})
    parquet_path = info.get("parquet", "")
    if not parquet_path:
        print(f"  {key:8s}: (trống)")
        continue
    df = pd.read_parquet(parquet_path)
    print(f"  {key:8s}: {len(df):>5d} dòng")
    print(f"    parquet: {parquet_path}")

print(f"\nRow counts: {news_paths.get('row_counts', {})}")

OUTPUT FILES
  rss     :   590 dòng
    parquet: D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news\rss\date=2026-05-03\PART-000.parquet
  html    :    96 dòng
    parquet: D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news\html\date=2026-05-03\PART-000.parquet

Row counts: {'rss': 590, 'html': 96}


## Validation & Data Quality

In [ ]:
# Validate per-source output with NEWS_COLUMNS
for key in ["rss", "html"]:
    info = news_paths.get(key, {})
    parquet_path = info.get("parquet", "")
    if not parquet_path:
        print(f"{key}: no output")
        continue

    df = pd.read_parquet(parquet_path)
    issues = validate_news_df(df)
    print(f"{key}: {len(df)} rows")
    if issues:
        print("  ⚠ issues:")
        for i in issues:
            print(f"    - {i}")
    else:
        print("  ✓ validation passed")
    print(f"  columns: {list(df.columns)}")

rss: 590 rows
  ✓ validation passed
  columns: ['article_id', 'source', 'ticker', 'title', 'summary', 'body_text', 'url', 'published_at', 'fetched_at', 'language', 'raw_ref']
html: 96 rows
  ✓ validation passed
  columns: ['article_id', 'source', 'ticker', 'title', 'summary', 'body_text', 'url', 'published_at', 'fetched_at', 'language', 'raw_ref']


In [ ]:
# Quick preview for sentiment-ready fields
for key in ["rss", "html"]:
    info = news_paths.get(key, {})
    parquet_path = info.get("parquet", "")
    if not parquet_path:
        continue
    df = pd.read_parquet(parquet_path)
    print(f"\n[{key}] preview")
    print(df[["article_id", "source", "ticker", "title", "body_text", "published_at"]].head(3))


[rss] preview
                                          article_id  \
0  40d4ca6978e8446cda496abdaf2e50cf42fa875f4f6e50...   
1  9459b809303b848e06330cd8e2aacabf67ab12a434117d...   
2  f4d32f49cd4d5f176e3a8966fe8fb3c779ebacf99c0502...   

                     source ticker  \
0  vnexpress_kinh_doanh_rss    NaN   
1  vnexpress_kinh_doanh_rss    NaN   
2  vnexpress_kinh_doanh_rss    NaN   

                                               title body_text  \
0  Từ 25/5, chủ nhà trọ thu tiền điện sai có thể ...             
1                            OPEC+ tăng sản xuất dầu             
2        Giá tăng nhưng thị trường NFT vẫn kém nhiệt             

               published_at  
0 2026-05-03 14:02:19+00:00  
1 2026-05-03 10:56:13+00:00  
2 2026-05-03 06:53:12+00:00  

[html] preview
                                          article_id          source ticker  \
0  40d4ca6978e8446cda496abdaf2e50cf42fa875f4f6e50...  vnexpress_html    NaN   
1  490274125285c3074b295ac3e4670b4594e7e7eb262ff

In [ ]:
# Quality report: full crawled news data (all dates by default)
import pandas as pd
from pathlib import Path

news_root = Path(getattr(news_cfg, "news_root", Path("data-lake/raw/Unstructure_Data/news")))
target_run_date = getattr(news_cfg, "run_date", None)
include_all_dates = True  # True => tổng quan toàn bộ lịch sử date=* dưới news_root

if not news_root.exists():
    raise FileNotFoundError(f"news_root không tồn tại: {news_root}")

sources_from_paths = []
if "news_paths" in globals() and isinstance(news_paths, dict):
    sources_from_paths = [
        k for k, v in news_paths.items()
        if k != "row_counts" and isinstance(v, dict)
    ]

sources_from_dirs = sorted([p.name for p in news_root.iterdir() if p.is_dir()])
sources = sorted(set(sources_from_paths + sources_from_dirs))

def _collect_parquet_files(src: str) -> list[Path]:
    src_root = news_root / src
    if not src_root.exists():
        return []
    if include_all_dates or not target_run_date:
        return sorted(src_root.glob("date=*/PART-*.parquet"))
    return sorted((src_root / f"date={target_run_date}").glob("PART-*.parquet"))

def _extract_partition_date(path: Path) -> str:
    for part in path.parts:
        if part.startswith("date="):
            return part.replace("date=", "")
    return "unknown"

def _safe_to_datetime(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, errors="coerce", utc=True)

report_rows = []
frames = []

for src in sources:
    files = _collect_parquet_files(src)
    if not files:
        report_rows.append(
            {
                "source": src,
                "file_count": 0,
                "rows": 0,
                "date_count": 0,
                "unique_article_id": 0,
                "duplicate_article_id": 0,
                "unique_tickers": 0,
                "null_article_id": None,
                "null_title": None,
                "null_url": None,
                "null_published_at": None,
                "min_published_at": None,
                "max_published_at": None,
                "min_fetched_at": None,
                "max_fetched_at": None,
            }
        )
        continue

    parts = []
    for fp in files:
        part = pd.read_parquet(fp)
        part["__file"] = str(fp)
        part["__partition_date"] = _extract_partition_date(fp)
        parts.append(part)

    df_src = pd.concat(parts, ignore_index=True, sort=False)
    frames.append(df_src.assign(__source_group=src))

    id_series = df_src["article_id"] if "article_id" in df_src.columns else pd.Series([], dtype="object")
    id_non_null = id_series.dropna().astype(str)
    duplicate_ids = int(id_non_null.duplicated().sum())

    pub = _safe_to_datetime(df_src["published_at"]) if "published_at" in df_src.columns else pd.Series(dtype="datetime64[ns, UTC]")
    fet = _safe_to_datetime(df_src["fetched_at"]) if "fetched_at" in df_src.columns else pd.Series(dtype="datetime64[ns, UTC]")

    report_rows.append(
        {
            "source": src,
            "file_count": len(files),
            "rows": int(len(df_src)),
            "date_count": int(df_src["__partition_date"].nunique()) if "__partition_date" in df_src.columns else 0,
            "unique_article_id": int(id_non_null.nunique()),
            "duplicate_article_id": duplicate_ids,
            "unique_tickers": int(df_src["ticker"].dropna().astype(str).nunique()) if "ticker" in df_src.columns else 0,
            "null_article_id": int(df_src["article_id"].isna().sum()) if "article_id" in df_src.columns else int(len(df_src)),
            "null_title": int(df_src["title"].isna().sum()) if "title" in df_src.columns else int(len(df_src)),
            "null_url": int(df_src["url"].isna().sum()) if "url" in df_src.columns else int(len(df_src)),
            "null_published_at": int(pub.isna().sum()) if "published_at" in df_src.columns else int(len(df_src)),
            "min_published_at": str(pub.min()) if not pub.empty and pub.notna().any() else None,
            "max_published_at": str(pub.max()) if not pub.empty and pub.notna().any() else None,
            "min_fetched_at": str(fet.min()) if not fet.empty and fet.notna().any() else None,
            "max_fetched_at": str(fet.max()) if not fet.empty and fet.notna().any() else None,
        }
    )

quality_df = pd.DataFrame(report_rows)
if not quality_df.empty:
    quality_df = quality_df.sort_values(by=["rows", "source"], ascending=[False, True], kind="stable").reset_index(drop=True)

print("=" * 90)
print("QUALITY REPORT — CRAWLED NEWS")
print("=" * 90)
print(f"news_root       : {news_root}")
print(f"target_run_date : {target_run_date} (include_all_dates={include_all_dates})")
display(quality_df)

if frames:
    all_df = pd.concat(frames, ignore_index=True, sort=False)
    available_cols = [c for c in NEWS_COLUMNS if c in all_df.columns]
    missing_cols = [c for c in NEWS_COLUMNS if c not in all_df.columns]

    all_ids = all_df["article_id"].dropna().astype(str) if "article_id" in all_df.columns else pd.Series([], dtype="object")
    dup_all_ids = int(all_ids.duplicated().sum())

    print("-" * 90)
    print("OVERALL SUMMARY")
    print("-" * 90)
    print(f"total_rows           : {len(all_df):,}")
    print(f"unique_article_id    : {all_ids.nunique():,}")
    print(f"duplicate_article_id : {dup_all_ids:,}")
    print(f"total_sources        : {all_df['__source_group'].nunique() if '__source_group' in all_df.columns else 0:,}")
    print(f"total_partitions     : {all_df['__partition_date'].nunique() if '__partition_date' in all_df.columns else 0:,}")
    print(f"available_schema     : {len(available_cols)}/{len(NEWS_COLUMNS)} columns")
    if missing_cols:
        print(f"missing_columns      : {missing_cols}")
    else:
        print("missing_columns      : []")

    if "__partition_date" in all_df.columns:
        by_date = (
            all_df.groupby("__partition_date", as_index=False)
            .agg(
                rows=("article_id", "size"),
                unique_article_id=("article_id", lambda s: s.dropna().astype(str).nunique()),
            )
            .sort_values("__partition_date", ascending=True)
            .rename(columns={"__partition_date": "partition_date"})
        )

        if "ticker" in all_df.columns:
            tickers_by_date = (
                all_df.groupby("__partition_date")["ticker"]
                .apply(lambda s: s.dropna().astype(str).nunique())
                .reset_index(name="unique_tickers")
                .rename(columns={"__partition_date": "partition_date"})
            )
            by_date = by_date.merge(tickers_by_date, on="partition_date", how="left")
        else:
            by_date["unique_tickers"] = 0

        by_source_date = (
            all_df.groupby(["__partition_date", "__source_group"], as_index=False)
            .agg(rows=("article_id", "size"))
            .sort_values(["__partition_date", "rows"], ascending=[True, False])
            .rename(columns={"__partition_date": "partition_date", "__source_group": "source"})
        )

        print("-" * 90)
        print("DAILY OVERVIEW (ALL CRAWLED DATES)")
        display(by_date)

        print("-" * 90)
        print("DAILY x SOURCE OVERVIEW")
        display(by_source_date)

    if "news_paths" in globals() and isinstance(news_paths, dict) and not include_all_dates:
        expected = news_paths.get("row_counts", {})
        if isinstance(expected, dict) and expected:
            compare_rows = []
            for src in quality_df["source"].tolist():
                expected_rows = int(expected.get(src, 0))
                actual_rows = int(quality_df.loc[quality_df["source"] == src, "rows"].iloc[0])
                compare_rows.append(
                    {
                        "source": src,
                        "expected_row_counts": expected_rows,
                        "actual_rows_loaded": actual_rows,
                        "match": expected_rows == actual_rows,
                    }
                )
            print("-" * 90)
            print("CHECK row_counts vs parquet loaded (current run only)")
            display(pd.DataFrame(compare_rows))
else:
    print("Không có parquet nào để kiểm tra. Hãy chạy cell ingest trước hoặc kiểm tra lại news_root.")

QUALITY REPORT — CRAWLED NEWS
news_root       : D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news
target_run_date : 2026-05-03 (include_all_dates=True)


,source,file_count,rows,date_count,unique_article_id,duplicate_article_id,unique_tickers,null_article_id,null_title,null_url,null_published_at,min_published_at,max_published_at,min_fetched_at,max_fetched_at
0,rss,4,2482,4,2095,387,8,0,0,0,0,2026-04-11 12:23:00+00:00,2026-05-03 15:45:00+00:00,2026-04-18 11:41:47.219840+00:00,2026-05-03 15:58:08.414876+00:00
1,html,4,394,4,282,112,12,0,0,0,394,NaN,NaN,2026-04-18 11:42:20.490204+00:00,2026-05-03 15:58:35.518453+00:00


------------------------------------------------------------------------------------------
OVERALL SUMMARY
------------------------------------------------------------------------------------------
total_rows           : 2,876
unique_article_id    : 2,214
duplicate_article_id : 662
total_sources        : 2
total_partitions     : 4
available_schema     : 11/11 columns
missing_columns      : []
------------------------------------------------------------------------------------------
DAILY OVERVIEW (ALL CRAWLED DATES)


,partition_date,rows,unique_article_id,unique_tickers
0,2026-04-18,739,687,2
1,2026-04-20,722,675,4
2,2026-04-22,729,671,10
3,2026-05-03,686,637,5


------------------------------------------------------------------------------------------
DAILY x SOURCE OVERVIEW


,partition_date,source,rows
1,2026-04-18,rss,640
0,2026-04-18,html,99
3,2026-04-20,rss,624
2,2026-04-20,html,98
5,2026-04-22,rss,628
4,2026-04-22,html,101
7,2026-05-03,rss,590
6,2026-05-03,html,96
